### Deep Research Agent

- Using tools, structured outputs and hosted tools

In [1]:
# Import libraries
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown

In [2]:
load_dotenv(override=True)

True

#### OpenAI Hosted Tools

OpenAI Agent SDK includes the following hosted tools:
- The WebSearchTool lets an agent search the web.
- The FileSearchTool allows retrieving information from your OpenAI Vector Stores.
- The ComputerTool allows automating computer use tasks like taking screenshots and clicking.

In [3]:
# Define the search agent
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succinctly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [4]:
# Run the agent
message = "Latest AI Agent frameworks in 2026"

with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

In 2026, several AI agent frameworks have emerged, each catering to specific development needs:

- **LangGraph**: Specializes in complex, stateful workflows using graph-based state machines, offering durable execution and a robust ecosystem. ([awesomeagents.ai](https://awesomeagents.ai/tools/best-ai-agent-frameworks-2026/?utm_source=openai))

- **CrewAI**: Focuses on multi-agent role-based teams, supporting over 25 LLM providers out of the box, and is compliant with FedRAMP High standards. ([awesomeagents.ai](https://awesomeagents.ai/tools/best-ai-agent-frameworks-2026/?utm_source=openai))

- **Agno**: Formerly known as Phidata, Agno is designed for production multi-agent systems with a control plane, emphasizing scalability and management. ([awesomeagents.ai](https://awesomeagents.ai/tools/best-ai-agent-frameworks-2026/?utm_source=openai))

- **PydanticAI**: Provides type-safe structured agents, ensuring reliability and maintainability in agent development. ([awesomeagents.ai](https://awesomeagents.ai/tools/best-ai-agent-frameworks-2026/?utm_source=openai))

- **Semantic Kernel**: Developed by Microsoft, it integrates with Azure, offering enterprise solutions for AI agent development. ([awesomeagents.ai](https://awesomeagents.ai/tools/best-ai-agent-frameworks-2026/?utm_source=openai))

- **OpenAI Agents SDK**: Tailored for OpenAI's models, this SDK facilitates the creation of lightweight agents within OpenAI's ecosystem. ([awesomeagents.ai](https://awesomeagents.ai/tools/best-ai-agent-frameworks-2026/?utm_source=openai))

- **Claude Agent SDK**: Anthropic's framework for building agents compatible with their Claude models, focusing on safety and alignment. ([awesomeagents.ai](https://awesomeagents.ai/tools/best-ai-agent-frameworks-2026/?utm_source=openai))

Additionally, **MARS (Modular Agent with Reflective Search)** introduces a framework optimized for autonomous AI research, balancing performance with execution costs through budget-aware planning and modular construction. ([arxiv.org](https://arxiv.org/abs/2602.02660?utm_source=openai))

The landscape of AI agent frameworks in 2026 is diverse, with each offering unique features tailored to different development requirements. 

#### We'll now use structured outputs and include a description of the fields

In [5]:
# Define the planner agent
HOW_MANY_SEARCHES = 3

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query")

    query: str = Field(description="The search term to use for the web search")

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query")

planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)

In [6]:
# Run the agent
message = "Latest AI Agent frameworks in 2026"

with trace("Search"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)

searches=[WebSearchItem(reason='To gather information on the most popular and widely used AI agent frameworks in 2026.', query='latest AI agent frameworks 2026'), WebSearchItem(reason='To find insights and reviews on emerging AI agent technologies and their applications in 2026.', query='emerging AI agents technology 2026'), WebSearchItem(reason='To explore the features and capabilities of AI agent frameworks released or updated in 2026.', query='AI agent frameworks features 2026')]


In [7]:
# Define a function tool to send email with the search results
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("iwanttotestanapp@gmail.com") # Change this to your verified email
    to_email = To("ctrlplusstyle@gmail.com") # Change this to your email
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return "success"

In [8]:
send_email

FunctionTool(name='send_email', description='Send out an email with the given subject and HTML body', params_json_schema={'properties': {'subject': {'title': 'Subject', 'type': 'string'}, 'html_body': {'title': 'Html Body', 'type': 'string'}}, 'required': ['subject', 'html_body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x76826ed4c050>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [9]:
# Define the email agent
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="EmailAgent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model="gpt-4o-mini",
)

In [10]:
# Define the report writing agent
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)

class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the research findings")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")

writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o",
    output_type=ReportData
)

#### The next 3 functions will plan and execute the search, using planner_agent and search_agent

In [13]:
# Define the main function to run the agents together
async def plan_searches(query: str):
    """Use the planner_agent to plan which searches to run for the query"""
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """Call search() for each item in the search plan"""
    print("Search...")
    num_completed = 0
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """Use the search agent to run a web search for each item in the search plan"""
    search_input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, search_input)
    return result.final_output

#### The next 2 functions write a report and email it

In [14]:
# Define the main function to run the agents together
async def write_report(query: str, search_results: list[str]):
    """Use the writer agent to write a report based on the search results"""
    print("Thinking about the report...")
    report_input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, report_input)
    print("Finished writing report")
    return result.final_output

async def run_send_email(report: ReportData):
    """Use the email agent to send an email with the report"""
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report


In [15]:
# Now we can run the whole thing together
query = "Latest AI Agent frameworks in 2026"

with trace("Research Agent"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await run_send_email(report)
    print("Hooray!")

Starting research...
Planning searches...
Will perform 3 searches
Search...
Finished searching
Thinking about the report...
Finished writing report
Writing email...
Email sent
Hooray!
